# 02 — Assertions and Golden Outputs

## Why this notebook exists

In **`01_why_agents_are_hard_to_test.ipynb`** we saw how a naive `assert agent(x) == "expected"` unit test flakes the moment an agent varies its phrasing — and we named the three forces behind that fragility: non-determinism, no single ground truth, and multi-step compounding failures. The closing insight was: *"We can't `assert equals` our way out of this. We need graders that score outputs, not match them exactly."*

This notebook builds those graders — the cheapest, most reliable layer of the eval stack. Each grader is a plain callable that accepts an example and a candidate output and returns a `Score`. No LLM, no external service, no API key required.

## What you'll learn

- The `Score` dataclass — the shared return type every grader in this series produces.
- When **exact match** is appropriate (structured outputs, deterministic pipelines) and why it's brittle for free-text.
- How **contains** and **regex** graders give you partial-match power without LLM overhead.
- How to validate **structured outputs** with `pydantic` v2 — confirming an agent's JSON parses into the expected schema, and capturing the validation error when it doesn't.
- The discipline of **golden outputs**: storing a known-good string to a file and diffing against it — and why keeping goldens from silently going stale is as important as writing them.
- How all four grader families share a single `(example, output) -> Score` signature, so they compose transparently in notebook 03's harness.

## 1. Setup

All graders in this notebook share one return type: `Score`. Defining it here, once, as a `dataclass` gives us a consistent interface that notebook 03 will rely on when it builds the harness.

**Compatibility note:** In this notebook examples are plain dicts — `{"input": ..., "expected": ...}` — and graders read `example["expected"]`. Notebook 03 will formalize `Example` as a proper dataclass with `.input` and `.expected` attributes, at which point `example["expected"]` becomes `example.expected`. The grader signatures don't change — only how you access the field. This is called out again at each grader definition below.

If `pydantic` isn't installed yet:
```bash
pip install pydantic
```

In [1]:
import re
import tempfile
from dataclasses import dataclass
from pathlib import Path

from pydantic import BaseModel, ValidationError
from typing import Literal

# Track every temp file written in this notebook for cleanup at the end.
_temp_files: list[Path] = []


@dataclass
class Score:
    key: str            # grader identifier, e.g. "exact_match"
    score: float        # normalized to [0.0, 1.0]
    passed: bool
    comment: str = ""


print("Setup OK")
print(f"Score fields: {[f.name for f in Score.__dataclass_fields__.values()]}")

Setup OK
Score fields: ['key', 'score', 'passed', 'comment']


## 2. Exact / Contains / Regex Matching

The three cheapest graders form a spectrum:

| Grader | Passes when | Best for |
|---|---|---|
| `exact_match` | `output == expected` exactly | Structured strings, enum labels, short deterministic answers |
| `make_contains(substring)` | `substring in output` | Checking that a required phrase or field name appears anywhere |
| `make_regex(pattern)` | `re.search(pattern, output)` matches | Format constraints — dates, codes, capitalization, numeric ranges |

**Exact match is brittle for free text** — a single trailing space or synonym flips it to fail. Its strength is precisely that brittleness: when an output *should* be deterministic (a classification label, a formatted code), exact match punishes any deviation.

Each grader accepts `(example, output)` where `example` is a dict `{"input": ..., "expected": ...}`. The `example["expected"]` field is used by `exact_match`; the contains/regex graders ignore `expected` and grade the raw output against the substring/pattern they were constructed with. In notebook 03, `example` becomes an `Example` dataclass and `example["expected"]` becomes `example.expected` — the grader body doesn't change.

In [2]:
# ── Grader 1: exact_match ────────────────────────────────────────────────────

def exact_match(example: dict, output: str) -> Score:
    """Pass iff output equals example['expected'] exactly."""
    expected = example["expected"]
    passed = output == expected
    return Score(
        key="exact_match",
        score=1.0 if passed else 0.0,
        passed=passed,
        comment="" if passed else f"expected {expected!r}, got {output!r}",
    )


# ── Grader 2: make_contains ──────────────────────────────────────────────────

def make_contains(substring: str, key: str = "contains"):
    """Factory: return a grader that passes when `substring` appears in output."""
    def grader(example: dict, output: str) -> Score:
        passed = substring in output
        return Score(
            key=key,
            score=1.0 if passed else 0.0,
            passed=passed,
            comment="" if passed else f"{substring!r} not found in output",
        )
    grader.__name__ = key
    return grader


# ── Grader 3: make_regex ─────────────────────────────────────────────────────

def make_regex(pattern: str, key: str = "regex"):
    """Factory: return a grader that passes when `pattern` matches anywhere in output."""
    compiled = re.compile(pattern)
    def grader(example: dict, output: str) -> Score:
        match = compiled.search(output)
        passed = match is not None
        return Score(
            key=key,
            score=1.0 if passed else 0.0,
            passed=passed,
            comment="" if passed else f"pattern {pattern!r} not found in output",
        )
    grader.__name__ = key
    return grader


print("Graders defined: exact_match, make_contains, make_regex")

Graders defined: exact_match, make_contains, make_regex


In [3]:
### Try it

# Stubbed agent outputs — no API call needed.
output_correct   = "positive"
output_wrong     = "Positive"   # different capitalisation — exact_match will catch this
output_verbose   = "The sentiment is positive, with high confidence."

example = {"input": "The food was great!", "expected": "positive"}

# exact_match: pass and fail
s1 = exact_match(example, output_correct)
s2 = exact_match(example, output_wrong)
print(f"exact_match on {output_correct!r}: passed={s1.passed}, score={s1.score}")
print(f"exact_match on {output_wrong!r}:  passed={s2.passed}, comment={s2.comment!r}")
print()

# contains: checks that "positive" appears somewhere, even in a verbose answer
contains_positive = make_contains("positive", key="contains_positive")
s3 = contains_positive(example, output_verbose)
s4 = contains_positive(example, "The sentiment is negative.")
print(f"contains on verbose:   passed={s3.passed}")
print(f"contains on negative:  passed={s4.passed}, comment={s4.comment!r}")
print()

# regex: require the output to be exactly one of our label words (full-string match)
label_regex = make_regex(r"^(positive|negative|neutral)$", key="label_format")
s5 = label_regex(example, output_correct)
s6 = label_regex(example, output_verbose)
print(f"regex on {output_correct!r}:  passed={s5.passed}")
print(f"regex on verbose:     passed={s6.passed}, comment={s6.comment!r}")

exact_match on 'positive': passed=True, score=1.0
exact_match on 'Positive':  passed=False, comment="expected 'positive', got 'Positive'"

contains on verbose:   passed=True
contains on negative:  passed=False, comment="'positive' not found in output"

regex on 'positive':  passed=True
regex on verbose:     passed=False, comment="pattern '^(positive|negative|neutral)$' not found in output"


## 3. Structured-Output Validation with Pydantic

Exact match breaks the moment an agent's JSON output has an extra space or differently-ordered keys. What we actually care about is: *does the output represent a valid instance of the expected schema?*

`pydantic` v2's `model_validate_json` (for raw JSON strings) or `model_validate` (for dicts) raises a `ValidationError` with a precise error message when the output is invalid. We capture that message in `Score.comment` so a failing eval tells you *why* it failed — wrong field type, missing required field, value outside a `Literal`, etc.

`make_schema_grader(model)` is a factory: pass it a pydantic `BaseModel` subclass and get back a grader. The grader tries to parse the output; if validation succeeds, `passed=True`; otherwise `passed=False` and `comment` carries the first validation error.

In [4]:
# ── Pydantic model used as the schema under test ─────────────────────────────

class Classification(BaseModel):
    label: Literal["positive", "negative", "neutral"]
    confidence: float  # pydantic will enforce float; we don't constrain range here


# ── Grader 4: make_schema_grader ─────────────────────────────────────────────

def make_schema_grader(model, key: str = "valid_schema"):
    """Factory: return a grader that passes when `output` is valid JSON for `model`.

    Tries model.model_validate_json(output) first (raw JSON string); if `output`
    is already a dict, falls back to model.model_validate(output).
    On ValidationError, passed=False and comment holds the error detail.
    """
    def grader(example: dict, output) -> Score:
        try:
            if isinstance(output, str):
                model.model_validate_json(output)
            else:
                model.model_validate(output)
            return Score(key=key, score=1.0, passed=True)
        except ValidationError as exc:
            first_error = exc.errors(include_url=False)[0]
            comment = f"{first_error['loc']}: {first_error['msg']}"
            return Score(key=key, score=0.0, passed=False, comment=comment)
        except Exception as exc:
            return Score(key=key, score=0.0, passed=False, comment=str(exc))
    grader.__name__ = key
    return grader


schema_grader = make_schema_grader(Classification, key="valid_schema")
print("Classification model and schema_grader defined")

Classification model and schema_grader defined


In [5]:
### Try it

example = {"input": "Review text here.", "expected": None}  # schema grader ignores expected

# Pass: well-formed JSON matching the schema
valid_output = '{"label": "positive", "confidence": 0.92}'
s1 = schema_grader(example, valid_output)
print(f"valid JSON:          passed={s1.passed}, score={s1.score}")

# Fail: wrong label value (not in Literal)
bad_label = '{"label": "POSITIVE", "confidence": 0.92}'
s2 = schema_grader(example, bad_label)
print(f"wrong label case:    passed={s2.passed}, comment={s2.comment!r}")

# Fail: missing required field
missing_field = '{"label": "negative"}'
s3 = schema_grader(example, missing_field)
print(f"missing confidence:  passed={s3.passed}, comment={s3.comment!r}")

# Fail: malformed JSON
malformed = "label=positive confidence=0.9"
s4 = schema_grader(example, malformed)
print(f"malformed JSON:      passed={s4.passed}, comment={s4.comment!r}")

valid JSON:          passed=True, score=1.0
wrong label case:    passed=False, comment="('label',): Input should be 'positive', 'negative' or 'neutral'"
missing confidence:  passed=False, comment="('confidence',): Field required"
malformed JSON:      passed=False, comment='(): Invalid JSON: expected value at line 1 column 1'


## 4. Golden Outputs

A golden output is simply a **known-good agent response stored to a file**. The grader reads the file and checks for exact equality. This is powerful for long, structured outputs (multi-paragraph summaries, formatted reports) where you want to detect *any* change — including formatting drift — without rewriting the expected string in code every time.

The discipline has two parts:

1. **Writing goldens intentionally.** A golden file is a contract: this is what the agent should produce. Don't auto-generate golden files from the current agent output without reviewing them first.
2. **Updating goldens intentionally.** When the agent legitimately improves, update the golden and commit the diff. If goldens are silently auto-updated by CI, they stop catching regressions.

> **Gotcha:** Silent golden rot. If your build pipeline regenerates goldens automatically on every run, a regressed agent "passes" forever. The golden file should be a checked-in artifact reviewed at PR time, not an output artifact.

In [6]:
# ── Write a known-good golden file ───────────────────────────────────────────

GOLDEN_CONTENT = (
    "Summary: The product received overwhelmingly positive reviews. "
    "Customers highlighted fast shipping, durable packaging, and responsive support."
)

golden_fd = tempfile.NamedTemporaryFile(
    mode="w",
    suffix="_golden.txt",
    prefix="evals02_",
    delete=False,
)
golden_path = Path(golden_fd.name)
golden_fd.write(GOLDEN_CONTENT)
golden_fd.close()
_temp_files.append(golden_path)

print(f"Golden file written: {golden_path}")
print(f"Content: {golden_path.read_text()!r}")
print()


# ── Grader 5: make_golden_grader ─────────────────────────────────────────────

def make_golden_grader(golden_path, key: str = "golden"):
    """Factory: return a grader that passes when output equals the golden file's content exactly."""
    path = Path(golden_path)
    def grader(example: dict, output: str) -> Score:
        expected = path.read_text()
        passed = output == expected
        return Score(
            key=key,
            score=1.0 if passed else 0.0,
            passed=passed,
            comment="" if passed else (
                f"output differs from golden ({golden_path}): "
                f"first diff at char {next((i for i,(a,b) in enumerate(zip(expected, output)) if a != b), min(len(expected), len(output)))}"
            ),
        )
    grader.__name__ = key
    return grader


golden_grader = make_golden_grader(golden_path, key="golden")
print("make_golden_grader defined")

Golden file written: /var/folders/7x/f0bvpy6d4xs2zzlxprhq8p8h0000gn/T/evals02_hgszs2xr_golden.txt
Content: 'Summary: The product received overwhelmingly positive reviews. Customers highlighted fast shipping, durable packaging, and responsive support.'

make_golden_grader defined


In [7]:
### Try it

example = {"input": "Summarize the reviews.", "expected": None}  # golden grader reads from file

# Pass: output matches the golden exactly
s1 = golden_grader(example, GOLDEN_CONTENT)
print(f"exact match:      passed={s1.passed}, score={s1.score}")

# Fail: output has a minor mutation (trailing period changed to ellipsis)
mutated_output = GOLDEN_CONTENT.replace("support.", "support...")
s2 = golden_grader(example, mutated_output)
print(f"mutated output:   passed={s2.passed}")
print(f"  comment: {s2.comment}")

# Fail: output truncated
truncated_output = GOLDEN_CONTENT[:60]
s3 = golden_grader(example, truncated_output)
print(f"truncated output: passed={s3.passed}")
print(f"  comment: {s3.comment}")

exact match:      passed=True, score=1.0
mutated output:   passed=False
  comment: output differs from golden (/var/folders/7x/f0bvpy6d4xs2zzlxprhq8p8h0000gn/T/evals02_hgszs2xr_golden.txt): first diff at char 142
truncated output: passed=False
  comment: output differs from golden (/var/folders/7x/f0bvpy6d4xs2zzlxprhq8p8h0000gn/T/evals02_hgszs2xr_golden.txt): first diff at char 60


In [8]:
### Updating a golden intentionally

# Suppose the agent has legitimately improved. The workflow is:
#   1. Review the new output and confirm it is better.
#   2. Overwrite the golden file.
#   3. Commit the diff — the PR diff shows exactly what changed.

new_golden_content = (
    "Summary: The product received overwhelmingly positive reviews. "
    "Customers highlighted fast shipping, durable packaging, and responsive support. "
    "No recurring complaints were found."
)

# Simulate the intentional update:
golden_path.write_text(new_golden_content)
print(f"Golden updated. New content: {golden_path.read_text()!r}")
print()

# Old output now fails (correct — it no longer matches the improved golden):
s_old = golden_grader(example, GOLDEN_CONTENT)
print(f"old output vs updated golden: passed={s_old.passed}")

# New output passes:
s_new = golden_grader(example, new_golden_content)
print(f"new output vs updated golden: passed={s_new.passed}, score={s_new.score}")

Golden updated. New content: 'Summary: The product received overwhelmingly positive reviews. Customers highlighted fast shipping, durable packaging, and responsive support. No recurring complaints were found.'

old output vs updated golden: passed=False
new output vs updated golden: passed=True, score=1.0


## 5. Each Grader Returns a `Score` — They Compose

Every grader we defined — `exact_match`, `make_contains(...)`, `make_regex(...)`, `make_schema_grader(...)`, `make_golden_grader(...)` — has the same signature:

```python
grader(example: dict, output) -> Score
```

That uniformity is the point. Notebook 03 will build a `run_eval(agent, dataset, graders)` harness that iterates over a list of `(input, expected)` examples, calls the agent, and then passes the same `(example, output)` pair to each grader in the list — no special-casing needed.

Here we preview that composition with a single output and a mixed grader list.

In [9]:
### Try it — run all graders over one output

# A classification agent output we want to evaluate multiple ways simultaneously.
output_under_test = '{"label": "positive", "confidence": 0.87}'
example_under_test = {
    "input": "The checkout experience was smooth and fast.",
    "expected": '{"label": "positive", "confidence": 0.87}',
}

graders = [
    exact_match,
    make_contains("positive", key="contains_positive"),
    make_regex(r'"confidence":\s*0\.\d+', key="has_confidence_field"),
    make_schema_grader(Classification, key="valid_schema"),
    # golden_grader is omitted here since the golden file holds a different domain's output;
    # in notebook 03 each example will carry its own grader list.
]

print(f"{'key':<30} {'passed':<8} {'score':<8} comment")
print("-" * 72)
for grader in graders:
    s = grader(example_under_test, output_under_test)
    print(f"{s.key:<30} {str(s.passed):<8} {s.score:<8.1f} {s.comment}")

key                            passed   score    comment
------------------------------------------------------------------------
exact_match                    True     1.0      
contains_positive              True     1.0      
has_confidence_field           True     1.0      
valid_schema                   True     1.0      


## What you just learned

- **`Score`** — `key / score / passed / comment` — is the single return type every grader in this series produces. Defining it once means graders, harnesses, and reporters share a language.
- **`exact_match`** is unforgiving and valuable: it detects any deviation in a deterministic output, including case differences, trailing whitespace, or synonym substitution.
- **`make_contains`** and **`make_regex`** give partial-match power without LLM cost — useful for checking that required phrases, field names, or format constraints appear in a longer output.
- **`make_schema_grader`** (pydantic v2) validates structured output: it confirms the output is valid JSON that conforms to your schema, and when it doesn't, `Score.comment` tells you which field failed and why.
- **`make_golden_grader`** catches any change — including formatting drift — against a stored known-good reference. The discipline: write goldens intentionally, update them intentionally, commit the diff.
- All five grader families share `(example, output) -> Score`. That signature is the contract notebook 03 depends on.

## What's missing

We now have five graders, but we've been calling them one-by-one against hand-written stub outputs. In the real world you have a *dataset* — a list of `(input, expected)` pairs — and you want to run every grader over every example automatically, aggregate the results, and surface a pass rate.

**`03_building_an_eval_harness.ipynb`** does exactly that. It formalizes the `Example` dataclass (`.input` and `.expected` attributes replacing the `example["expected"]` dict access you've seen here), builds a `run_eval(agent, dataset, graders) -> results` runner, adds aggregate metrics (pass rate, mean score, per-grader breakdown), and presents the results as a readable table. The graders you defined here plug in without modification.

## Cleanup

The temp golden files written in Section 4 live in the OS scratch directory. Deleting them here keeps a fresh-kernel re-run idempotent.

In [10]:
removed: list[str] = []
for path in list(_temp_files):
    try:
        path.unlink()
        removed.append(str(path))
    except FileNotFoundError:
        pass
_temp_files.clear()

print(f"Removed {len(removed)} temp file(s):")
for r in removed:
    print(f"  - {r}")

Removed 1 temp file(s):
  - /var/folders/7x/f0bvpy6d4xs2zzlxprhq8p8h0000gn/T/evals02_hgszs2xr_golden.txt
